# Unstructured Parser 실습 


---

### 

1. **STEP 0 은 터미널(PowerShell)에서** 실행합니다. 노트북 안에서 실행하지 마세요.
2. STEP 1 부터는 셀을 **위에서 아래로 순서대로** 실행합니다.
3. 셀 왼쪽 `[ ]` 안에 숫자가 뜨면 실행이 끝난 것입니다. `[*]` 는 아직 실행 중입니다.
4. 에러가 나면 **메시지 맨 아랫줄부터** 읽으세요. 거기에 원인이 있습니다.

> 💡 `hi_res` 전략은 첫 실행 때 AI 모델을 내려받습니다. 몇 분 걸려도 정상입니다.


---
# STEP 0. Poetry 환경 설정

> ⚠️ **이 단계는 노트북이 아니라 PowerShell에서 실행합니다.**

### Poetry 가 뭔가요?

> 📦 **비유: Poetry 는 "프로젝트 전용 짐 보관소 관리인"입니다.**
>
> `pip` 로 라이브러리를 깔면 내 컴퓨터 전체에 쌓입니다. A 프로젝트는 버전 1이 필요하고 B 프로젝트는 버전 2가 필요하면 충돌이 나죠.
>
> Poetry 는 프로젝트마다 **독립된 보관소(.venv)** 를 만들고, 무엇을 어떤 버전으로 넣었는지 **장부(pyproject.toml / poetry.lock)** 에 적어둡니다. 나중에 다른 사람이 그 장부만 받으면 **똑같은 환경이 그대로 재현**됩니다.
>
> 포트폴리오에서 "환경 재현 가능합니다"라고 말할 수 있게 해주는 도구입니다.

### 0-1. Poetry 버전 확인

```powershell
poetry --version
```

### 0-2. 프로젝트 폴더 만들기

```powershell
mkdir C:\dev\rag-parser
cd C:\workspaces\rag-parser
```

> 경로에 **한글과 띄어쓰기를 넣지 마세요.** 바탕 화면(`C:\Users\홍길동\바탕 화면`)은 피합니다.

### 0-3. Poetry 프로젝트 초기화

```powershell
poetry init --no-interaction --name rag-parser --python ">=3.10,<3.13"
```

### 0-4. 가상환경을 프로젝트 폴더 안에 만들기

```powershell
poetry config virtualenvs.in-project true
```

이렇게 하면 `C:\dev\rag-parser\.venv` 폴더가 생깁니다. 어디에 설치됐는지 눈으로 볼 수 있어  안전합니다.

### 0-5. 라이브러리 설치

```powershell
poetry add "unstructured[pdf]" langchain-unstructured unstructured-client python-dotenv pymupdf pypdf
poetry add --group dev jupyterlab ipykernel
```

| 패키지 | 역할 |
| --- | --- |
| `unstructured[pdf]` | 문서를 Element 조각으로 분해 |
| `langchain-unstructured` | 파싱 결과를 LangChain Document 로 변환 |
| `unstructured-client` | API 방식으로 서버에 보낼 때 사용 |
| `python-dotenv` | API 키를 `.env` 파일에서 안전하게 읽기 |
| `pymupdf` | 스캔본 PDF 만들기 / PDF 유형 판별 |
| `pypdf` | 비교 실험용 단순 파서 |

### 0-6. 주피터 커널 등록

```powershell
poetry run python -m ipykernel install --user --name rag-parser --display-name "Python (rag-parser)"
```

### 0-7. 주피터 실행

```powershell
poetry run jupyter lab
```

브라우저가 열리면 이 노트북 파일을 열고, 우측 상단 커널을 **Python (rag-parser)** 로 바꿉니다.

---

### 경로 A(로컬 `hi_res`)를 쓸 경우에만 추가 설치

로컬에서 `hi_res` / `ocr_only` 를 돌리려면 외부 프로그램 두 개가 더 필요합니다.

1. **Poppler** — `oschwartz10612/poppler-windows` 릴리스 zip → `C:\poppler` 에 압축 해제
2. **Tesseract** — `UB-Mannheim/tesseract` 설치 파일 실행, 설치 중 **Korean 언어팩 체크**
3. 시스템 환경 변수 `Path` 에 아래 두 줄 추가 후 **PowerShell 재시작**

```
C:\poppler\Library\bin
C:\Program Files\Tesseract-OCR
```

4. 확인

```powershell
pdftoppm -v
tesseract --version
tesseract --list-langs
```




## 0-8. 환경 점검

아래 셀을 실행해 환경이 제대로 잡혔는지 확인합니다.
세 줄 모두 `OK` 가 나오면 다음 단계로 넘어가세요.

In [ ]:
import sys
import importlib
from pathlib import Path

# 1) 파이썬 버전을 출력하세요 (sys.version 사용)
# TODO

# 2) 지금 쓰는 인터프리터 경로를 출력하고, .venv 가 포함되는지 확인하세요
#    힌트: sys.executable
# TODO

# 3) 아래 패키지들이 설치됐는지 반복문으로 확인하세요
#    힌트: importlib.import_module(name) 을 try/except ImportError 로 감쌉니다
packages = [
    "unstructured",
    "langchain_unstructured",
    "unstructured_client",
    "dotenv",
    "fitz",      # pymupdf 의 실제 import 이름
    "pypdf",
]
# TODO

---
# STEP 1. 실습 파일 준비

실습에는 성격이 다른 PDF **3종**이 필요합니다.

| 파일 | 성격 | 확인 방법 |
| --- | --- | --- |
| `sample_text.pdf` | 텍스트 레이어가 살아있는 문서 | 글자가 마우스로 드래그됨 |
| `sample_table.pdf` | 표가 많은 문서 | 재무제표처럼 격자가 빽빽함 |
| `sample_scan.pdf` | 스캔본 (글자가 그림) | 글자가 드래그되지 않음 |

> 🖨️ **비유**: 텍스트 PDF가 **워드 파일**이라면, 스캔본은 그걸 **인쇄해서 복사기로 복사한 것**입니다.
> 글자는 보이지만 더 이상 긁어서 복사할 수 없는 '그림'이 되어버린 상태죠.
> 그래서 `fast` 전략은 빈 결과를 뱉고 `ocr_only` 가 필요해집니다.

## 1-1. 폴더 만들고 샘플 문서 받기

`sample_text.pdf` 는 Unstructured 공식 예제 파일을 씁니다.
공식 문서의 모든 예제가 이 파일 기준이라 막혔을 때 비교하기 좋습니다.

In [ ]:
import urllib.request
from pathlib import Path

# 1) ./data 폴더를 만드세요 (이미 있으면 그냥 넘어가야 합니다)
#    힌트: Path("./data").mkdir(exist_ok=True)
# TODO

# 2) 아래 URL 의 PDF 를 ./data/sample_text.pdf 로 내려받으세요
#    힌트: urllib.request.urlretrieve(URL, TARGET)
URL = ("https://raw.githubusercontent.com/Unstructured-IO/unstructured"
       "/main/example-docs/pdf/layout-parser-paper.pdf")
# TODO

# 3) 파일 크기를 KB 단위로 출력하세요
#    힌트: TARGET.stat().st_size
# TODO

### `sample_table.pdf` 는 직접 구해 오세요

DART 전자공시에서 받는 것을 추천합니다. 한국어 재무제표 표가 들어 있어 실무 감각을 익히기 좋습니다.

```
https://dart.fss.or.kr
→ 회사명 검색 (예: 삼성전자)
→ 정기공시 → 사업보고서
→ 우측 상단 [다운로드] → PDF
```

받은 파일을 `./data/sample_table.pdf` 로 저장하세요.
**연결재무상태표**나 **요약재무정보** 페이지가 포함되게 하는 것이 좋습니다.

## 1-2. 스캔본 PDF 만들기

스캔본은 인터넷에서 딱 맞는 걸 찾기가 의외로 어렵습니다.
그래서 **직접 만듭니다.** 수업용으로는 이게 훨씬 좋습니다.

- 원본 텍스트를 이미 알고 있으니 **OCR 정확도를 정량 측정**할 수 있습니다
- 모든 수강생이 **같은 파일**로 실습하니 결과 비교가 깔끔합니다
- 저작권 문제가 없습니다

In [ ]:
import fitz   # PyMuPDF

SRC = DATA/"sample_text.pdf"
DST = DATA/"sample_scan.pdf"
DPI = 150   # 낮출수록 OCR 난이도가 올라갑니다

# 1) 원본 PDF 를 열고, 빈 PDF 를 새로 만드세요
#    힌트: fitz.open(SRC) / fitz.open()
# TODO

# 2) 원본의 각 페이지를 반복하면서
#    (a) 이미지로 렌더링하고      힌트: page.get_pixmap(dpi=DPI)
#    (b) 새 PDF 에 페이지를 추가해 그 이미지를 붙이세요
#        힌트: dst_doc.new_page(width=..., height=...) / new_page.insert_image(...)
# TODO

# 3) 새 PDF 를 저장하고 두 문서를 닫으세요
# TODO

## 1-3. 세 파일이 정말 그 유형이 맞는지 검증

겉보기로는 구분이 안 됩니다. **반드시 코드로 확인**하세요.

In [ ]:
import fitz

def inspect(path, pages=3):
    """PDF 앞 몇 페이지에서 뽑히는 글자 수로 유형을 판별합니다."""
    # 1) PDF 를 열고 앞 pages 장의 글자 수를 모두 더하세요
    #    힌트: doc[i].get_text() 의 길이를 합산
    # TODO

    # 2) 글자 수가 50 미만이면 "스캔본", 아니면 "텍스트 PDF" 로 판정해 돌려주세요
    # TODO

# 3) 세 파일에 대해 함수를 호출하고 결과를 보기 좋게 출력하세요
#    파일이 없는 경우도 안내 메시지를 내도록 처리합니다
for name in ["sample_text.pdf", "sample_table.pdf", "sample_scan.pdf"]:
    pass  # TODO

---
# STEP 2. 실습 1 — 첫 파싱

드디어 문서를 **Element 조각**으로 분해합니다.

> 🍱 **비유: Unstructured 는 "문서계의 이삿짐 정리 전문가"입니다.**
>
> 짐을 트럭에 그냥 쏟아부으면 새 집에서 숟가락 하나 못 찾습니다.
> 정리 전문가는 박스마다 '그릇', '책', '옷' 이라고 **라벨**을 붙여주죠.
>
> PyPDF 로 PDF 를 읽으면 결과가 **긴 문자열 하나**입니다.
> Unstructured 는 **리스트**를 주고, 각 원소에 `Title` · `NarrativeText` · `Table` 같은 **라벨**이 붙어 있습니다.

In [ ]:
from unstructured.partition.auto import partition

# 1) ./data/sample_text.pdf 를 strategy="fast" 로 파싱하세요
#    languages 에는 ["kor", "eng"] 를 넣습니다
#    힌트: partition(filename=..., strategy=..., languages=...)
elements = None  # TODO

# 2) 생성된 Element 개수를 출력하세요
# TODO

## 2-1. 카테고리 분포 보기

**파싱 품질을 가장 빠르게 눈으로 점검하는 방법**입니다.
`UncategorizedText` 비율이 높으면 파싱이 실패하고 있다는 신호입니다.

In [ ]:
from collections import Counter

# 1) 모든 Element 의 category 를 세어 분포를 구하세요
#    힌트: Counter(el.category for el in elements)
distribution = None  # TODO

# 2) 분포를 보기 좋게 출력하세요 (most_common() 활용)
# TODO

# 3) UncategorizedText 가 전체에서 차지하는 비율(%)을 계산해 출력하세요
#    이 값이 파싱 품질 지표입니다
# TODO

## 2-2. Element 안을 들여다보기

각 Element 에는 **본문 + 라벨 + 출처 메타데이터**가 함께 붙어 있습니다.

> 🎯 **취업 연결 포인트**
> 챗봇 답변에 "📎 출처: 사업계획서.pdf 12페이지" 를 표시하면 면접관 반응이 달라집니다.
> 이게 가능한 이유가 바로 `page_number` 메타데이터입니다. 포트폴리오에 꼭 넣으세요.

In [ ]:
# 1) 앞에서 5개 Element 만 골라 아래 세 가지를 출력하세요
#    - 카테고리            힌트: el.category
#    - 본문 앞 60글자       힌트: el.text[:60]
#    - 파일명과 페이지 번호  힌트: el.metadata.filename / el.metadata.page_number
# TODO

# 2) 첫 번째 Element 의 메타데이터에 어떤 항목이 들어있는지 전부 출력하세요
#    힌트: elements[0].metadata.to_dict()
# TODO

## 2-3. 라벨의 힘 — 표만 골라내기

`el.category == "Table"` **한 줄**로 표만 골라냅니다.

PyPDF 였다면 불가능했습니다. 긴 문자열에서 "여기부터 저기까지가 표"라는 걸 알아낼 방법이 없거든요.
정규표현식으로 어떻게든 해보려다 밤새는 분들이 많습니다.

In [ ]:
# 1) 같은 PDF 를 strategy="hi_res", infer_table_structure=True 로 다시 파싱하세요
#    (표 구조를 살리려면 이 두 옵션이 필요합니다)
table_elements = None  # TODO

# 2) category 가 "Table" 인 Element 만 리스트로 골라내세요
#    힌트: 리스트 컴프리헨션 한 줄이면 됩니다
tables = None  # TODO

# 3) 표를 몇 개 찾았는지 출력하고,
#    첫 번째 표의 HTML 을 앞 400자만 출력하세요
#    힌트: tables[0].metadata.text_as_html
# TODO

---
# STEP 3. 파싱 전략 비교

`strategy` 파라미터 하나가 **속도와 품질을 모두** 결정합니다.

| 전략 | 동작 방식 | 속도 | 정확도 | 적합한 문서 |
| --- | --- | --- | --- | --- |
| `fast` | 전통 NLP 기법으로 텍스트만 빠르게 추출 | 매우 빠름 | 보통 | 텍스트 레이어가 살아있는 PDF |
| `hi_res` | 비전 모델이 레이아웃을 인식 | 느림 | 매우 높음 | 표·다단·복잡한 보고서 |
| `ocr_only` | OCR 로만 추출 | 느림 | 높음 | 스캔본, 사진 문서 |
| `auto` | 문서 특성에 따라 자동 선택 | 가변 | 가변 | 첫 테스트 |

> 🔍 **비유: 사진을 보는 두 가지 방법**
> - `fast` = **복사-붙여넣기.** PDF 안의 글자 데이터를 그대로 긁어옵니다. 빠르지만 "이게 제목인지 표인지"는 모릅니다.
> - `hi_res` = **사람이 눈으로 보고 판단.** AI 비전 모델이 문서를 그림처럼 보고 "여기 네모 박스는 표구나" 하고 레이아웃을 인식합니다.

### 판단 순서 3문장 

1. PDF 를 열어 **글자가 드래그되는가** → 되면 `fast` 부터 시도
2. **표가 중요하거나 2단 편집**인가 → `hi_res`
3. **글자가 드래그되지 않는가** → 이미지 스캔본이므로 `ocr_only`

In [ ]:
import time

def benchmark(path, strategy, **kwargs):
    """전략 하나를 실행하고 소요 시간과 품질 지표를 돌려줍니다."""
    # 1) 시작 시각을 기록하세요        힌트: time.perf_counter()
    # TODO

    # 2) partition 을 실행하세요 (에러가 나도 멈추지 않도록 try/except 로 감쌉니다)
    # TODO

    # 3) 소요 시간을 계산하고, 아래 항목을 담은 딕셔너리를 돌려주세요
    #    - strategy   : 전략 이름
    #    - seconds    : 소요 시간(소수 2자리)
    #    - elements   : Element 개수
    #    - tables     : Table 개수
    #    - unknown_%  : UncategorizedText 비율(%)
    # TODO


# 4) sample_text.pdf 에 대해 fast 와 hi_res 를 각각 측정하고
#    결과를 표 형태로 출력하세요
# TODO

## 3-1. 스캔본에 `fast` 를 쓰면 어떻게 될까

**직접 눈으로 확인하세요.** 이 한 번의 실험이 전략 선택의 중요성을 가장 잘 설명합니다.

In [ ]:
scan = DATA / "sample_scan.pdf"

# 1) 스캔본을 strategy="fast" 로 파싱하고
#    Element 개수와 추출된 전체 글자 수를 출력하세요
#    (거의 0에 가까울 것입니다 — 왜 그럴까요?)
# TODO

# 2) 같은 스캔본을 strategy="ocr_only" 로 파싱하고 똑같이 출력하세요
#    ※ Tesseract 가 없으면 에러가 납니다. try/except 로 감싸고 안내 메시지를 내세요
# TODO

# 3) 두 결과의 차이를 한 문장으로 적어보세요
#    내 답변:

---
# STEP 4. 실습 2 — LangChain 연동과 청킹

파싱 결과를 **LangChain Document 객체**로 받아 Vector DB 에 바로 넣을 수 있는 형태까지 만듭니다.

### 먼저 `.env` 파일을 만드세요

프로젝트 폴더(`C:\workspaces\rag-parser`)에 `.env` 파일을 만들고 아래 한 줄을 적습니다.

```
UNSTRUCTURED_API_KEY=여기에_발급받은_키를_붙여넣기
```

같은 폴더에 `.gitignore` 도 만들고 아래를 적습니다.

```
.env
.venv/
data/
__pycache__/
```

> 🔐 **왜 이렇게 하나요?**
> API 키를 코드에 직접 쓰면 GitHub 에 올라가는 순간 봇이 긁어갑니다. 실제로 몇 분 안에 긁힙니다.
> `.env` 는 **열쇠를 넣어두는 서랍**, `.gitignore` 는 **"이 서랍은 이삿짐에 싣지 마세요"** 라는 메모입니다.
> 둘은 **항상 세트**로 만드세요.
>
> Part 3 에서 Vercel 에 배포할 때도 똑같은 원리입니다. 거기선 Vercel 대시보드의 Environment Variables 가 서랍 역할을 합니다.

In [ ]:
import os
from dotenv import load_dotenv

# 1) .env 파일을 읽어 환경변수로 올리세요
#    힌트: load_dotenv()
# TODO

# 2) UNSTRUCTURED_API_KEY 값을 가져오세요
API_KEY = None  # TODO

# 3) 키가 있으면 앞 4자리와 뒤 4자리만 출력하고,
#    없으면 무엇을 확인해야 하는지 안내 메시지를 출력하세요
#    ※ 키 전체를 출력하면 노트북 공유 시 유출됩니다. 절대 전체를 찍지 마세요
# TODO

## 4-1. UnstructuredLoader 로 한 번에 불러오기

로더 하나로 **파싱 + 청킹**이 동시에 끝납니다.

- `partition_via_api=True` → 무거운 모델을 내 PC 가 아닌 **서버에서** 실행
- 로컬로만 돌리고 싶으면 `api_key` 와 `partition_via_api` 두 줄을 지우면 됩니다

In [ ]:
from langchain_unstructured import UnstructuredLoader

# 1) UnstructuredLoader 를 아래 조건으로 만드세요
#    - file_path              : ./data/sample_text.pdf
#    - api_key                : 위에서 읽은 API_KEY
#    - partition_via_api      : True
#    - strategy               : "hi_res"
#    - chunking_strategy      : "by_title"
#    - max_characters         : 1500
#    - combine_text_under_n_chars : 200
loader = None  # TODO

# 2) load() 로 Document 리스트를 받고 개수를 출력하세요
docs = None  # TODO

In [ ]:
# 앞에서 3개 청크를 꺼내 아래를 출력하세요
#  - 청크 번호
#  - 본문 앞 250자        힌트: doc.page_content
#  - 출처 파일명과 페이지   힌트: doc.metadata.get("filename") / .get("page_number")
#  - 청크 길이
# TODO

## 4-2. `by_title` 청킹이 왜 중요한가

> 📚 **비유: 책을 자르는 두 가지 방법**
> - **글자 수 분할** = 자로 재서 100페이지마다 칼로 자르기 → 문장이 두 동강
> - **`by_title`** = 챕터 제목이 나올 때마다 챕터 단위로 묶기 → 의미가 보존됨

RAG 에서 검색이 자꾸 엉뚱한 답을 준다면, **임베딩 모델을 바꾸기 전에 청킹부터 의심**하세요.
끊어진 문장은 임베딩도 엉망으로 나옵니다.

In [ ]:
import statistics

def load_and_measure(chunking):
    # 1) chunking 인자가 있으면 chunking_strategy 로 넣고,
    #    없으면 청킹 없이 로더를 만드세요
    # TODO

    # 2) load() 결과에서 각 청크 길이를 모아
    #    개수 / 평균 / 최소 / 최대 길이를 담은 딕셔너리를 돌려주세요
    # TODO


# 3) 청킹 없음 vs by_title 을 비교해 표로 출력하세요
# TODO

# 4) 두 결과를 보고 알게 된 점을 한 문장으로 적어보세요
#    내 답변:

---
# STEP 5. 과제 — 파서 비교 실험 리포트

**[권장 난이도: 중급]** · **[추천 소요 시간: 2~3시간]** · **[교육 과정 주차: 12주차]**

문서 3종을 파서 3종으로 각각 파싱하고, **측정값을 근거로** 선택을 설명하는 리포트를 작성합니다.

| 측정 항목 | 의미 |
| --- | --- |
| 처리 소요 시간(초) | 실서비스 응답 속도와 직결 |
| 추출 글자 수 | 내용을 얼마나 건졌는가 |
| 표 보존 여부 | 구조 인식 능력 |
| `UncategorizedText` 비율 | 파싱 실패 신호 |

> 💪 기업이 보고 싶은 건 "라이브러리를 써봤다"가 아니라
> **"문제를 정의하고, 실험하고, 근거로 선택했다"** 는 사고 과정입니다.

In [ ]:
import time
from pypdf import PdfReader

def run_pypdf(path):
    # 1) PyPDF 로 전체 텍스트를 추출하고 소요 시간을 재세요
    #    힌트: PdfReader(path).pages 를 돌며 page.extract_text()
    #    표 개수는 0 으로 고정합니다 (PyPDF 는 표를 인식하지 못합니다)
    # TODO


def run_unstructured(path, strategy):
    # 2) Unstructured 로 파싱하고 아래를 측정하세요
    #    - seconds   : 소요 시간
    #    - chars     : 추출된 전체 글자 수
    #    - tables    : Table Element 개수
    #    - unknown_% : UncategorizedText 비율
    #    ※ 스캔본 + fast 처럼 실패하는 조합이 있으니 try/except 로 감싸세요
    # TODO


# 3) 문서 3종 × 파서 3종을 모두 돌려 결과를 rows 리스트에 모으세요
#    각 결과에는 어떤 문서였는지도 함께 기록합니다
rows = []
# TODO

## 5-1. 결과를 표로 정리하고 저장하기

측정 결과를 **README 에 바로 붙일 수 있는 마크다운 표**로 만듭니다.

In [ ]:
import csv

COLS = ["document", "parser", "seconds", "chars", "tables", "unknown_%"]

# 1) rows 를 마크다운 표 형태로 출력하세요
#    (README 에 그대로 붙여넣을 수 있어야 합니다)
# TODO

# 2) 같은 내용을 benchmark_result.csv 로 저장하세요
#    힌트: csv.DictWriter, encoding="utf-8-sig" (엑셀 한글 깨짐 방지)
# TODO

---
## 5-2. 결론 작성

> 측정은 끝났습니다. 이제 **해석**이 남았습니다.
> 숫자를 나열하는 건 누구나 합니다. **"그래서 무엇을 선택했고 왜인가"** 가 포트폴리오의 핵심입니다.

아래 빈칸을 채워 README 에 넣으세요.

```markdown
## 파서 선택 근거

### 실험 설계
- 문서: 텍스트 PDF / 표 중심 PDF / 스캔본 PDF 각 1종
- 파서: PyPDF, Unstructured(fast), Unstructured(hi_res)
- 측정: 처리 시간, 추출 글자 수, 표 보존, 분류 실패 비율

### 결과
(위에서 출력한 마크다운 표를 여기에 붙여넣기)

### 해석
- 텍스트 PDF 에서는 ______ 가 ______ 배 빨랐지만, ______ 는 보존하지 못했다.
- 표 중심 문서에서는 ______ 만 표를 ___개 인식했다.
- 스캔본에서는 ______ 가 사실상 아무것도 추출하지 못했다 (글자 ___자).

### 결론
우리 서비스는 ______ 한 문서를 주로 다루므로 ______ 를 선택했다.
다만 ______ 한 경우에는 ______ 로 분기 처리한다.

### 트러블슈팅
- 증상: ______
- 원인: ______
- 해결: ______
```

---

## ✅ 제출 체크리스트

- [ ] 이 노트북이 **처음부터 끝까지 에러 없이** 실행된다
- [ ] 문서 3종 × 파서 3종 측정 결과가 표로 정리되어 있다
- [ ] 파서 선택의 **정량적 근거**(측정값)를 제시했다
- [ ] 파싱 전/후 데이터를 스크린샷으로 비교했다
- [ ] 전처리 파이프라인을 아키텍처 다이어그램으로 시각화했다
- [ ] 트러블슈팅 사례를 1건 이상 기록했다
- [ ] API 키가 코드에 하드코딩되지 않고 `.env` 로 관리된다
- [ ] `.gitignore` 에 `.env` 가 포함되어 있다
- [ ] `pyproject.toml` 과 `poetry.lock` 이 저장소에 함께 올라가 있다

---

## 다음 차시 예고

| 차시 | 주제 | 오늘 내용과의 연결 |
| --- | --- | --- |
| 13주차 | 임베딩과 벡터 DB | 오늘 만든 청크를 임베딩해 저장 |
| 14주차 | 검색 품질 튜닝 | 청킹 전략이 검색 정확도에 주는 영향 측정 |
| Part 3 | MCP 와 배포 | `.env` 관리가 Vercel 환경변수로 확장 |

**오늘의 한 문장: "RAG 튜닝은 임베딩이 아니라 파싱에서 시작한다."**
